In [1]:
import cv2
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from keras.models import Sequential
from keras.layers import Dense, Conv2D, MaxPooling2D, Flatten, Dropout
from tensorflow.keras.preprocessing.image import ImageDataGenerator

In [2]:
train_gen=ImageDataGenerator(rescale=1./255)
train_data=train_gen.flow_from_directory('afhq/train',target_size=(128,128),batch_size=32,class_mode='categorical',shuffle=True) 
test_gen=ImageDataGenerator(rescale=1./255)
test_data=test_gen.flow_from_directory('afhq/val',target_size=(128,128),batch_size=1,class_mode='categorical',shuffle=False)

Found 14630 images belonging to 3 classes.
Found 1500 images belonging to 3 classes.
Found 1500 images belonging to 3 classes.


In [3]:
# Create a Sequential model
classifier_model = Sequential()

# Convolutional layers
classifier_model.add(Conv2D(32, (3, 3), activation='relu', input_shape=(128, 128, 3), name="line"))
classifier_model.add(MaxPooling2D((2, 2)))
classifier_model.add(Conv2D(64, (3, 3), activation='relu', name="segment"))
classifier_model.add(MaxPooling2D((2, 2)))
classifier_model.add(Conv2D(128, (3, 3), activation='relu', name="region"))
classifier_model.add(MaxPooling2D((2, 2)))
classifier_model.add(Conv2D(256, (3, 3), activation='relu', name="surface"))
classifier_model.add(MaxPooling2D((2, 2)))
classifier_model.add(Conv2D(512, (3, 3), activation='relu', name="contour"))
classifier_model.add(MaxPooling2D((2, 2)))

# Flatten the feature maps
classifier_model.add(Flatten())

# Fully connected layers
classifier_model.add(Dense(128, activation='relu', name="face"))
classifier_model.add(Dropout(0.5))  # Dropout for regularization
classifier_model.add(Dense(3, activation='softmax', name="species"))  # Adjust the output size based on the number of classes

c:\Program Files\Python312\Lib\site-packages\keras\src\layers\convolutional\base_conv.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [4]:
classifier_model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ line (Conv2D)                   │ (None, 126, 126, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 63, 63, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ segment (Conv2D)                │ (None, 61, 61, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 30, 30, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ region (Conv2D)                 │ (None, 28, 28, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 14, 14, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ surface (Conv2D)                │ (None, 12, 12, 256)    │       295,168 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_3 (MaxPooling2D)  │ (None, 6, 6, 256)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ contour (Conv2D)                │ (None, 4, 4, 512)      │     1,180,160 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_4 (MaxPooling2D)  │ (None, 2, 2, 512)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 2048)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ face (Dense)                    │ (None, 128)            │       262,272 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ species (Dense)                 │ (None, 3)              │           387 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,831,235 (6.99 MB)

 Trainable params: 1,831,235 (6.99 MB)

 Non-trainable params: 0 (0.00 B)

In [5]:
classifier_model.compile(loss='categorical_crossentropy',optimizer='adam',metrics=['accuracy'])

In [6]:
classifier_model.fit(train_data,epochs=10,batch_size = 64)

c:\Program Files\Python312\Lib\site-packages\keras\src\trainers\data_adapters\py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


Epoch 1/10
458/458 ━━━━━━━━━━━━━━━━━━━━ 196s 425ms/step - accuracy: 0.5652 - loss: 0.8352
Epoch 2/10
458/458 ━━━━━━━━━━━━━━━━━━━━ 196s 425ms/step - accuracy: 0.5652 - loss: 0.8352
Epoch 2/10
458/458 ━━━━━━━━━━━━━━━━━━━━ 87s 190ms/step - accuracy: 0.9443 - loss: 0.1650
Epoch 3/10
458/458 ━━━━━━━━━━━━━━━━━━━━ 87s 190ms/step - accuracy: 0.9443 - loss: 0.1650
Epoch 3/10
458/458 ━━━━━━━━━━━━━━━━━━━━ 86s 187ms/step - accuracy: 0.9684 - loss: 0.0956
Epoch 4/10
458/458 ━━━━━━━━━━━━━━━━━━━━ 86s 187ms/step - accuracy: 0.9684 - loss: 0.0956
Epoch 4/10
458/458 ━━━━━━━━━━━━━━━━━━━━ 86s 189ms/step - accuracy: 0.9801 - loss: 0.0565
Epoch 5/10
458/458 ━━━━━━━━━━━━━━━━━━━━ 86s 189ms/step - accuracy: 0.9801 - loss: 0.0565
Epoch 5/10
458/458 ━━━━━━━━━━━━━━━━━━━━ 86s 188ms/step - accuracy: 0.9856 - loss: 0.0426
Epoch 6/10
458/458 ━━━━━━━━━━━━━━━━━━━━ 86s 188ms/step - accuracy: 0.9856 - loss: 0.0426
Epoch 6/10
458/458 ━━━━━━━━━━━━━━━━━━━━ 86s 188ms/step - accuracy: 0.9891 - loss: 0.0328
458/458 ━━━━━━━━━━━

In [7]:
classifier_model.summary() 

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ line (Conv2D)                   │ (None, 126, 126, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 63, 63, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ segment (Conv2D)                │ (None, 61, 61, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 30, 30, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ region (Conv2D)                 │ (None, 28, 28, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 14, 14, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ surface (Conv2D)                │ (None, 12, 12, 256)    │       295,168 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_3 (MaxPooling2D)  │ (None, 6, 6, 256)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ contour (Conv2D)                │ (None, 4, 4, 512)      │     1,180,160 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_4 (MaxPooling2D)  │ (None, 2, 2, 512)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 2048)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ face (Dense)                    │ (None, 128)            │       262,272 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ species (Dense)                 │ (None, 3)              │           387 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 5,493,707 (20.96 MB)

 Trainable params: 1,831,235 (6.99 MB)

 Non-trainable params: 0 (0.00 B)

 Optimizer params: 3,662,472 (13.97 MB)

In [8]:
pred=classifier_model.predict(test_data).argmax(axis=1)

1500/1500 ━━━━━━━━━━━━━━━━━━━━ 17s 11ms/step
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 17s 11ms/step


In [9]:
from sklearn.metrics import classification_report

In [10]:
print(classification_report(pred ,test_data.classes ))

              precision    recall  f1-score   support

           0       0.99      0.98      0.99       503
           1       0.97      0.98      0.98       496
           2       0.98      0.98      0.98       501

    accuracy                           0.98      1500
   macro avg       0.98      0.98      0.98      1500
weighted avg       0.98      0.98      0.98      1500



In [11]:
classifier_model.save("animal_classifier_model.h5")